# Variance swap

Static replication of a variance swap with a strip of OTM options of a single maturity $T$, using 10 strikes from the 10d put to the 10d call:

$$\sigma^2_{\text{realized}} \approx \frac{2}{T}\int_0^{\infty} \frac{O(K)}{K^2}\,dK \;\;\Rightarrow\;\; w_i = \frac{2}{T}\,\frac{\Delta K_i}{K_i^2}$$

with $O(K)$ the OTM option (put for $K<F$, call for $K \ge F$) and $\Delta K_i = (K_{i+1}-K_{i-1})/2$ (one-sided at the ends).

**Proof**: the portfolio replicates the log contract, whose cash gamma is constant: $\Gamma \cdot S^2 = 2/T$ for every spot and every time. The scatter plot shows the portfolio gamma cash $\Gamma \cdot S^2 / 100$ (target $2/(100\,T)$) with spot on each strike for each time to maturity in `t_bd` (business days).

## Import

In [13]:
import numpy as np
import pandas as pd
from scipy.stats import norm
import plotly.graph_objects as go

## Custom functions

In [14]:
def bs_gamma(S, K, t, sigma, r=0.0, q=0.0):
    """Black-Scholes gamma (broadcast over S, K, sigma)."""
    sqrt_t = np.sqrt(t)
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * t) / (sigma * sqrt_t)
    return np.exp(-q * t) * norm.pdf(d1) / (S * sigma * sqrt_t)


def bs_delta(S, K, t, sigma, cp, r=0.0, q=0.0):
    """Black-Scholes spot delta, cp = +1 call / -1 put."""
    d1 = (np.log(S / K) + (r - q + 0.5 * sigma**2) * t) / (sigma * np.sqrt(t))
    return cp * np.exp(-q * t) * norm.cdf(cp * d1)


def variance_swap_weights(options, spot, T, r=0.0, q=0.0, t_bd=(20, 19, 18, 17, 16, 15), n_strikes=10, delta_cut=0.10, bd_per_year=252):
    """
    options   : DataFrame of a single maturity with columns 'strike', 'type' ('C'/'P'), 'iv'
    spot      : underlying spot
    T         : maturity in years
    t_bd      : times to maturity (in business days) at which the portfolio is evaluated in the proof
    n_strikes : number of strikes kept in the strip
    delta_cut : wings of the strip (10d put to 10d call by default)
    bd_per_year : business days per year to convert t_bd to years

    Returns (weights, fig):
      weights : DataFrame of the OTM options with their replication weight (per 1 unit of variance notional)
      fig     : plotly scatter of the portfolio gamma cash vs spot (= strikes), one trace per t
    """
    df = options.copy()
    df["type"] = df["type"].str.upper().str[0]
    F = spot * np.exp((r - q) * T)

    # keep OTM options: puts below the forward, calls at/above
    otm = ((df["type"] == "P") & (df["strike"] < F)) | ((df["type"] == "C") & (df["strike"] >= F))
    df = df[otm].sort_values("strike").reset_index(drop=True)

    # keep strikes between the 10d put and the 10d call
    cp = np.where(df["type"] == "C", 1.0, -1.0)
    df["delta"] = bs_delta(spot, df["strike"], T, df["iv"], cp, r, q)
    df = df[df["delta"].abs() >= delta_cut].reset_index(drop=True)

    # pick n_strikes evenly spaced across that range (wings included)
    idx = np.unique(np.round(np.linspace(0, len(df) - 1, n_strikes)).astype(int))
    df = df.iloc[idx].reset_index(drop=True)

    K = df["strike"].to_numpy(float)
    iv = df["iv"].to_numpy(float)
    dK = np.gradient(K)
    df["weight"] = 2.0 * dK / (T * K**2)
    weights = df[["strike", "type", "iv", "delta", "weight"]]

    # proof: portfolio gamma cash with spot on each strike, for each bumped time to maturity
    S = K[:, None]
    fig = go.Figure()
    for bd in t_bd:
        t = bd / bd_per_year
        gamma_cash = (bs_gamma(S, K[None, :], t, iv[None, :], r, q) * S**2 / 100) @ df["weight"].to_numpy()
        fig.add_trace(go.Scatter(x=K, y=gamma_cash, mode="lines+markers", name=f"t={bd}bd"))
    fig.add_hline(y=2 / (100 * T), line_dash="dash", annotation_text="target 2/(100T)")
    fig.update_layout(title="Variance swap portfolio gamma cash",
                      xaxis_title="Strike (spot level)", yaxis_title="Gamma cash  (Γ·S²/100)")
    return weights, fig

## Example

In [15]:
spot, T, r, q = 100.0, 21 / 252, 0.03, 0.0   # 1-month option (21 business days)
F = spot * np.exp((r - q) * T)

strikes = np.arange(70, 131, 1.0)
iv = 0.20 - 0.10 * np.log(strikes / F) + 0.30 * np.log(strikes / F) ** 2   # synthetic smile
options = pd.concat([
    pd.DataFrame({"strike": strikes, "type": "C", "iv": iv}),
    pd.DataFrame({"strike": strikes, "type": "P", "iv": iv}),
], ignore_index=True)

weights, fig = variance_swap_weights(options, spot, T, r, q, t_bd=[20, 19, 18, 17, 16, 15])
weights

,strike,type,iv,delta,weight
0,93.0,P,0.209198,-0.101458,0.005550
1,95.0,P,0.206247,-0.175339,0.003989
2,96.0,P,0.204895,-0.223028,0.003906
3,98.0,P,0.202425,-0.338072,0.003748
4,99.0,P,0.201302,-0.403218,0.003673
5,101.0,C,0.199272,0.459873,0.003529
6,102.0,C,0.198360,0.392220,0.003460
7,104.0,C,0.196732,0.268225,0.003328
8,105.0,C,0.196014,0.214828,0.003265
9,107.0,C,0.194758,0.129060,0.004193


In [16]:
fig.show()